# mindex perf — result explorer

Visualizes the per-run CSVs written by `perf/run.sh` (one file per run, one row per
concurrency level — see `README.md` → *Reading the results*).

**Usage:** point `RESULT_DIRS` (next cell) at one or more folders of `*.csv`, then
*Run All*. Every CSV under those folders is loaded and concatenated, so this compares
across runs / configs / machines. Rows are grouped into a **config signature**
(`eb`=embed_batch, `pool`=db_pool_size, `xb`=embedder_batch, `mi`=max_inflight, plus the
free-text `label`); within a signature, concurrency is the x-axis.

The CSVs themselves are gitignored (kept locally, never committed); only this notebook
is committed. Deps: `pandas`, `matplotlib` (`pip install pandas matplotlib`).

In [ ]:
# --- CONFIG: edit this list, then Run All -----------------------------------------
# Folders to scan for *.csv (relative to this notebook, or absolute). Add more to
# compare saved runs across machines/configs, e.g. ['results', 'archived/rocm-7900'].
RESULT_DIRS = ['results']

# Recurse into sub-folders when scanning each dir.
RECURSIVE = True

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()

# Numeric columns we coerce on load (everything not here stays string/categorical).
NUMERIC_COLS = [
    'embed_batch', 'db_pool_size', 'embedder_batch', 'embedder_max_inflight',
    'embedder_maxlen', 'concurrency', 'total_files', 'total_bytes', 'total_chunks',
    'wall_clock_s', 'chunks_per_s', 'files_per_s', 'mb_per_s',
    'req_dur_p50', 'req_dur_p90', 'req_dur_p95', 'req_dur_p99', 'http_reqs',
    'err_429', 'err_499', 'err_500', 'err_503', 'err_other',
    'fwd_batch_mean', 'fwd_batch_max', 'embedder_encode_s', 'queue_highwater',
    'embedder_429', 'min_pool_available',
]


def find_csvs(dirs, recursive=True):
    files = []
    for d in dirs:
        p = Path(d)
        if not p.is_absolute():
            p = NOTEBOOK_DIR / p
        if p.is_file() and p.suffix == '.csv':
            files.append(p)
            continue
        if not p.is_dir():
            print(f'  skip (not found): {p}')
            continue
        files.extend(sorted(p.rglob('*.csv') if recursive else p.glob('*.csv')))
    return sorted(set(files))


def load_results(dirs, recursive=True):
    frames = []
    for f in find_csvs(dirs, recursive):
        # NA-aware: run.sh writes the literal string 'NA' for missing fields.
        df = pd.read_csv(f, na_values=['NA', ''])
        df['source_file'] = f.name
        frames.append(df)
        print(f'  loaded {len(df):>3} rows  {f.name}')
    if not frames:
        raise FileNotFoundError(
            f'No CSVs under {dirs!r}. Run perf/run.sh first, or fix RESULT_DIRS.'
        )
    df = pd.concat(frames, ignore_index=True)
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    # Drop exact dupes (same run re-globbed from overlapping dirs).
    df = df.drop_duplicates(subset=[c for c in df.columns if c != 'source_file'])
    return df


def config_sig(row):
    """Stable per-config label; concurrency is varied *within* a config, so it's out."""
    def fmt(v):
        return 'NA' if pd.isna(v) else (str(int(v)) if float(v).is_integer() else str(v))
    sig = (f"eb{fmt(row.get('embed_batch'))}/pool{fmt(row.get('db_pool_size'))}"
           f"/xb{fmt(row.get('embedder_batch'))}/mi{fmt(row.get('embedder_max_inflight'))}")
    label = row.get('label')
    if isinstance(label, str) and label.strip():
        sig += f" [{label.strip()}]"
    return sig


df = load_results(RESULT_DIRS, RECURSIVE)
df['config'] = df.apply(config_sig, axis=1)
# Derived: average chunks carried per /index request — the real cap on forward-pass
# batch when shards are small (see the fwd_batch section below).
df['chunks_per_req'] = df['total_chunks'] / df['http_reqs']
# Derived: fraction of wall time the GPU actually spent encoding (1.0 ⇒ GPU-bound).
df['encode_ratio'] = df['embedder_encode_s'] / df['wall_clock_s']

print(f'\n{len(df)} rows across {df["config"].nunique()} config(s), '
      f'concurrency levels: {sorted(int(x) for x in df["concurrency"].dropna().unique())}')
configs = sorted(df['config'].unique())

## Summary table

The headline columns per (config, concurrency). `encode_ratio` near 1.0 = GPU-bound;
much lower = GPU idle waiting on slicing / Qdrant / transport. `chunks_per_req` ≈
`fwd_batch_mean` means the forward-pass batch is capped by shard size, not `--batch`.

In [ ]:
summary_cols = [
    'config', 'concurrency', 'chunks_per_s', 'wall_clock_s', 'embedder_encode_s',
    'encode_ratio', 'fwd_batch_mean', 'chunks_per_req', 'req_dur_p95',
    'err_429', 'err_500', 'err_503', 'min_pool_available',
]
summary = (df[[c for c in summary_cols if c in df.columns]]
           .sort_values(['config', 'concurrency'])
           .reset_index(drop=True))
summary.round(2)

In [ ]:
# Shared plot helpers: one line per config signature, x = concurrency.
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']


def series(cfg):
    """Rows for one config, sorted by concurrency (NaN-y/y dropped per-call)."""
    return df[df['config'] == cfg].sort_values('concurrency')


def by_concurrency(ax, ycol, ylabel, title, ref=None):
    """Plot ycol vs concurrency, one line per config. ref draws a dashed baseline."""
    for i, cfg in enumerate(configs):
        s = series(cfg).dropna(subset=['concurrency', ycol])
        if s.empty:
            continue
        ax.plot(s['concurrency'], s[ycol], marker=MARKERS[i % len(MARKERS)],
                label=cfg, linewidth=1.6)
    if ref is not None:
        ax.axhline(ref, ls='--', color='grey', lw=1, label=ref_label)
    ax.set_xlabel('concurrency (VUs)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if len(configs) > 1 or ref is not None:
        ax.legend(fontsize=8)
    # Integer concurrency ticks.
    xs = sorted(df['concurrency'].dropna().unique())
    if xs:
        ax.set_xticks(xs)

## 1. Throughput vs concurrency

Primary metric. Sublinear scaling = the embedder serializing on one GPU worker thread;
a flat tail = the throughput ceiling (more VUs only add latency past that point).

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
by_concurrency(a, 'chunks_per_s', 'chunks / s', 'Throughput (chunks/s) vs concurrency')
by_concurrency(b, 'mb_per_s', 'MB / s', 'Throughput (MB/s) vs concurrency')
fig.tight_layout()
plt.show()

## 2. Latency vs throughput (the knee = optimum)

Each point is one concurrency level (annotated). The **knee** — where p95 starts
climbing steeply while throughput barely moves — marks the config to pick.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for i, cfg in enumerate(configs):
    s = series(cfg).dropna(subset=['chunks_per_s', 'req_dur_p95'])
    if s.empty:
        continue
    ax.plot(s['chunks_per_s'], s['req_dur_p95'], marker=MARKERS[i % len(MARKERS)],
            label=cfg, linewidth=1.6)
    for _, r in s.iterrows():
        ax.annotate(f"c={int(r['concurrency'])}", (r['chunks_per_s'], r['req_dur_p95']),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('chunks / s')
ax.set_ylabel('p95 /index latency (ms)')
ax.set_title('Latency vs throughput — knee is the sweet spot')
ax.grid(True, alpha=0.3)
if len(configs) > 1:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 3. Latency percentiles vs concurrency

p50 / p90 / p95 / p99 of per-`/index` request duration as load rises.

In [ ]:
fig, axes = plt.subplots(1, len(configs), figsize=(6.2 * len(configs), 4.5), squeeze=False)
pcts = [('req_dur_p50', 'p50'), ('req_dur_p90', 'p90'),
        ('req_dur_p95', 'p95'), ('req_dur_p99', 'p99')]
for ax, cfg in zip(axes[0], configs):
    s = series(cfg).dropna(subset=['concurrency'])
    for col, lab in pcts:
        if col in s.columns:
            ax.plot(s['concurrency'], s[col], marker='o', label=lab, linewidth=1.5)
    ax.set_xlabel('concurrency (VUs)')
    ax.set_ylabel('latency (ms)')
    ax.set_title(cfg, fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    xs = sorted(s['concurrency'].dropna().unique())
    if xs:
        ax.set_xticks(xs)
fig.tight_layout()
plt.show()

## 4. GPU utilization — `encode_ratio` (= embedder_encode_s / wall_clock_s)

The single clearest "is the GPU busy" signal. **≈ 1.0 ⇒ GPU-bound** (wall time is almost
all encoding); **well below 1.0 ⇒ the GPU idles** waiting on the rest of the pipeline
(slicing / Qdrant upsert / transport), so concurrency still has room to fill it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ref_label = 'GPU-bound (encode ≈ wall)'
by_concurrency(ax, 'encode_ratio', 'encode_s / wall_s',
               'GPU encode share of wall time', ref=1.0)
ax.set_ylim(0, 1.1)
fig.tight_layout()
plt.show()

## 5. Effective forward-pass batch — is the GPU actually fed?

`fwd_batch_mean` is sequences per GPU forward pass. If it sits far **below** both the
embedder `--batch` and the chunks carried per request, the GPU is starved. The grey
line is the configured embedder `--batch` (the ceiling); the dashed line is
`chunks_per_req` (the corpus-sharding cap). When `fwd_batch_mean ≈ chunks_per_req ≪ --batch`,
the limiter is **shard size**, not the batch flag — pack more files per `/index` to grow it.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: fwd_batch_mean vs concurrency, with the --batch ceiling and chunks/req cap.
for i, cfg in enumerate(configs):
    s = series(cfg).dropna(subset=['concurrency', 'fwd_batch_mean'])
    if s.empty:
        continue
    m = MARKERS[i % len(MARKERS)]
    a.plot(s['concurrency'], s['fwd_batch_mean'], marker=m, label=f'{cfg} · fwd_mean', lw=1.6)
    if 'chunks_per_req' in s:
        a.plot(s['concurrency'], s['chunks_per_req'], marker=m, ls=':', lw=1.2,
               label=f'{cfg} · chunks/req', alpha=0.7)
    xb = s['embedder_batch'].dropna()
    if not xb.empty:
        a.axhline(xb.iloc[0], ls='--', color='grey', lw=1)
a.set_xlabel('concurrency (VUs)')
a.set_ylabel('sequences / forward pass')
a.set_title('Forward-pass batch vs configured --batch (grey)')
a.set_yscale('log')
a.grid(True, alpha=0.3, which='both')
a.legend(fontsize=7)
xs = sorted(df['concurrency'].dropna().unique())
if xs:
    a.set_xticks(xs)

# Right: throughput vs effective forward batch (does feeding the GPU more help?).
for i, cfg in enumerate(configs):
    s = series(cfg).dropna(subset=['fwd_batch_mean', 'chunks_per_s'])
    if s.empty:
        continue
    b.scatter(s['fwd_batch_mean'], s['chunks_per_s'],
              marker=MARKERS[i % len(MARKERS)], s=60, label=cfg)
b.set_xlabel('fwd_batch_mean (sequences / forward pass)')
b.set_ylabel('chunks / s')
b.set_title('Throughput vs effective forward-pass batch')
b.grid(True, alpha=0.3)
if len(configs) > 1:
    b.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 6. Backpressure & pool headroom

Where the concurrency ceiling shows up. **Errors** (`err_429` embedder/claim backpressure,
`err_500` SQLite pool exhaustion, `err_503` embedder down, `embedder_429` server-side
backpressure) climbing off zero, or **`min_pool_available`** hitting 0, marks the limit.
All zeros / full pool = plenty of headroom left.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))

err_cols = [c for c in ['err_429', 'err_499', 'err_500', 'err_503', 'err_other',
                        'embedder_429'] if c in df.columns]
agg = (df.dropna(subset=['concurrency'])
         .groupby('concurrency')[err_cols].sum())
if agg.values.sum() == 0:
    a.text(0.5, 0.5, 'No errors recorded\n(all err_* = 0)', ha='center', va='center',
           transform=a.transAxes, fontsize=12, color='green')
    a.set_title('Errors vs concurrency')
else:
    agg.plot(kind='bar', stacked=True, ax=a)
    a.set_ylabel('count (summed across configs)')
    a.set_title('Errors vs concurrency')
a.set_xlabel('concurrency (VUs)')
a.grid(True, alpha=0.3, axis='y')

by_concurrency(b, 'min_pool_available', 'min SQLite pool available',
               'Pool headroom (0 = saturated)')
pool = df['db_pool_size'].dropna()
if not pool.empty:
    b.axhline(pool.max(), ls='--', color='grey', lw=1, label=f'pool size = {int(pool.max())}')
    b.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Takeaways (auto-generated)

A few heuristics printed from the data — sanity hints, not a substitute for reading the
plots above.

In [ ]:
for cfg in configs:
    s = series(cfg).dropna(subset=['concurrency'])
    if s.empty:
        continue
    print(f'▸ {cfg}')
    best = s.loc[s['chunks_per_s'].idxmax()]
    print(f'   peak throughput {best["chunks_per_s"]:.1f} chunks/s @ c={int(best["concurrency"])}')
    er = s.dropna(subset=['encode_ratio'])
    if not er.empty:
        hi = er.loc[er['encode_ratio'].idxmax()]
        verdict = 'GPU-bound' if hi['encode_ratio'] >= 0.9 else 'GPU under-fed'
        print(f'   max GPU encode share {hi["encode_ratio"]:.0%} @ c={int(hi["concurrency"])}  → {verdict}')
    fb = s.dropna(subset=['fwd_batch_mean', 'embedder_batch', 'chunks_per_req'])
    if not fb.empty:
        r = fb.iloc[0]
        if r['fwd_batch_mean'] < 0.5 * r['embedder_batch'] and \
           abs(r['fwd_batch_mean'] - r['chunks_per_req']) < 0.25 * r['chunks_per_req'] + 2:
            print(f'   fwd_batch_mean ≈ {r["fwd_batch_mean"]:.0f} ≈ chunks/req '
                  f'(≪ --batch {int(r["embedder_batch"])}) → forward pass capped by SHARD SIZE, '
                  'not --batch; pack more files per /index to grow it.')
    errs = s[err_cols].sum().sum() if err_cols else 0
    mp = s['min_pool_available'].dropna()
    if errs == 0 and (mp.empty or mp.min() > 0):
        print('   no errors, pool never drained → concurrency headroom remains.')
    print()